# make observation plan

## 필요한 모듈

이 프로젝트를 위해서는 아래의 모듈이 필요하다. 

> numpy, pandas, matplotlib, astroquery, requests, astropy, certifi, version_information

### 모듈 설치

1. 콘솔 창에서 모듈을 설치할 때는 아래와 같은 형식으로 입력하면 된다.

>pip install module_name==version

>conda install module_name==version

2. 주피터 노트북(코랩 포함)에 설치 할 때는 아래의 셀을 실행해서 실행되지 않은 모듈을 설치할 수 있다. (pip 기준) 만약 아나콘다 환경을 사용한다면 7행을 콘다 설치 명령어에 맞게 수정하면 된다.

In [1]:
#import sys
#!pip install astropy==5.3

In [1]:
import importlib, sys, subprocess
packages = "numpy, pandas, matplotlib, astroquery, requests, astropy, certifi, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        print(f"**** module {pkg} is not installed")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    else: 
        print(f"**** module {pkg} is installed")

%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

**** module numpy is installed
**** module pandas is installed
**** module matplotlib is installed
**** module astroquery is installed
**** module requests is installed
**** module astropy is installed
**** module certifi is installed
**** module version_information is installed
This notebook was generated at 2023-10-09 17:45:05 (KST = GMT+0900) 
0 Python     3.11.5 64bit [GCC 11.2.0]
1 IPython    8.15.0
2 OS         Linux 5.15.0 86 generic x86_64 with glibc2.31
3 numpy      1.26.0
4 pandas     2.0.3
5 matplotlib 3.7.2
6 astroquery 0.4.6
7 requests   2.31.0
8 astropy    5.2.1
9 certifi    2023.07.22
10 version_information 1.0.4


### import modules

In [2]:
#from urllib.request import urlopen,quote    #남중할 때의 적경 적위 찾아내기(최대고도인 애들 위주로)
from datetime import datetime, timedelta, timezone
from astropy.time import Time

#from astropy.time import TimezoneInfo 
import astropy.units as u

import pandas as pd
import numpy as np

import sys
import json
import base64
import requests

In [3]:
dt_kst_now = datetime.now()
print(dt_kst_now)
timezone_utc = timezone(timedelta(hours=-9))
dt_utc_now = dt_kst_now.astimezone(timezone_utc)
print(dt_utc_now)

2023-10-09 17:45:13.172773
2023-10-08 23:45:13.172773-09:00


In [4]:
start_dt_kst_str = '2023-09-11T20:19:18'
end_dt_kst_str = '2023-12-31T00:00:00'

start_dt_kst = datetime.strptime(start_dt_kst_str, '%Y-%m-%dT%H:%M:%S')
print(start_dt_kst)
print(type(start_dt_kst))

start_dt_utc = start_dt_kst + timedelta(hours=-9)
print(start_dt_utc)
print(type(start_dt_utc))

start_dt_utc_str = start_dt_utc.strftime('%Y-%m-%dT%H:%M:%S')
print("start_dt_utc_str :", start_dt_utc_str) 
print("type(start_dt_utc_str) :", type(start_dt_utc_str))



2023-09-11 20:19:18
<class 'datetime.datetime'>
2023-09-11 11:19:18
<class 'datetime.datetime'>
start_dt_utc_str : 2023-09-11T11:19:18
type(start_dt_utc_str) : <class 'str'>


In [5]:
from astropy.time import Time

start_time_utc = Time(start_dt_utc_str, format='isot', scale='utc', location=('127d', '37d'))  # Loses timezone info, converts to UTC
print("start_time_utc", start_time_utc) 

start_time_utc

start_time_utc 2023-09-11T11:19:18.000


<Time object: scale='utc' format='isot' value=2023-09-11T11:19:18.000>

In [6]:
#%%
#############################################
# variables
mpc_code='P64'
obs_dt_utc = datetime.strptime(start_dt_utc_str, '%Y-%m-%dT%H:%M:%S')
print("obs_dt_utc :", obs_dt_utc)
print("type(obs_dt_utc) :", type(obs_dt_utc))
obs_datetime = obs_dt_utc.strftime('%Y-%m-%d %H:%M:%S')
elev_min=45
time_min=15
vmag_min=6
vmag_max=13
list_num=50
sort='trans'


obs_dt_utc : 2023-09-11 11:19:18
type(obs_dt_utc) : <class 'datetime.datetime'>


In [8]:
# Define API URL and SPK filename:
url = 'https://ssd-api.jpl.nasa.gov/sbwobs.api'
spk_filename = 'spk_file.bsp'

# Get the requested SPK-ID from the command-line:
if (len(sys.argv)) == 1:
  print("please specify SPK-ID on the command-line")
  sys.exit(2)
spkid = sys.argv[1]

In [9]:
# Build the appropriate URL for this API request:
# IMPORTANT: You must encode the "=" as "%3D" and the ";" as "%3B" in the
#            Horizons COMMAND parameter specification.
url += f"?sb-kind=a&mpc-code={mpc_code}&obs-time={str(obs_datetime)}&elev-min={elev_min}&time-min={time_min}"
url += f"&vmag-max={vmag_max}&vmag-min={vmag_min}&optical=true&fmt-ra-dec=true&mag-required=true&output-sort={sort}&maxoutput={list_num}"

print("url :", url)


url : https://ssd-api.jpl.nasa.gov/sbwobs.api?sb-kind=a&mpc-code=P64&obs-time=2023-09-11 11:19:18&elev-min=45&time-min=15&vmag-max=13&vmag-min=6&optical=true&fmt-ra-dec=true&mag-required=true&output-sort=trans&maxoutput=50


In [10]:
import requests
# Submit the API request and decode the JSON-response:
response = requests.get(url)
try:
  data = json.loads(response.text)
  print(data)
except ValueError:
  print("Unable to decode JSON results")

{'signature': {'version': '1.0', 'source': 'NASA/JPL Small-Body Observability API'}, 'obs_constraints': {'mag-required': 'true', 'elev-min': '45', 'time-min': '15', 'obs-time': '2023-09-11 11:19:18', 'maxoutput': '50', 'optical': 'true', 'vmag-max': '13', 'fmt-ra-dec': 'true', 'mpc-code': 'P64', 'vmag-min': '6', 'output-sort': 'trans'}, 'sb_constraints': {'sb-kind': 'a'}, 'location': 'GSHS Observatory, Suwon', 'obs_night': {'sun_set_az': '277.0', 'sun_set': '2023-09-10 09:47', 'end_civil': '2023-09-10 10:13', 'end_nautical': '2023-09-10 10:44', 'end_astronomical': '2023-09-10 11:16', 'begin_astronomical': '2023-09-10 19:39', 'begin_nautical': '2023-09-10 20:11', 'begin_civil': '2023-09-10 20:42', 'sun_rise_az': '83.2', 'sun_rise': '2023-09-10 21:08', 'transit_phase': '0.14', 'transit': '2023-09-11 00:41 ', 'moon_set_phase': '0.20', 'moon_set': '2023-09-10 07:35 ', 'moon_rise_phase': '0.16', 'moon_rise': '2023-09-10 17:03 ', 'begin_dark': '2023-09-10 11:16', 'mid_dark': '2023-09-10 15:2

In [11]:
type(data)
data['obs_constraints']
data.keys
data['sb_constraints']
data['fields']
data['data']
type(data['fields'])
len(data['fields'])
len(data['data'][0])

14

In [12]:
df = pd.DataFrame.from_dict(data['data'])
df = df.set_axis((data['fields']), axis=1)
df

,Designation,Full name,Rise time,Transit time,Set time,Max. time observable,R.A.,Dec.,Vmag,Helio. range (au),Topo.range (au),Object-Observer-Sun (deg),Object-Observer-Moon (deg),Galactic latitude (deg)
0,393,393 Lampetia (A894 VC),09:23*,11:16*,13:09,01:52,19:01:21,"+00 37'30""",11.5,1.87,1.19,116.50,149.5,-1.66
1,914,914 Palisana (A919 NC),08:52*,12:08,15:24,04:07,19:53:47,"+21 32'05""",12.2,1.96,1.21,124.18,133.6,-2.98
2,675,675 Ludmilla (A908 QH),11:48,13:07,14:27,02:39,20:53:06,"-03 40'47""",12.3,2.64,1.75,144.69,156.8,-28.39
3,133,133 Cyrene (A873 QA),14:02,14:28,14:54,00:51,22:14:09,"-07 16'53""",12.5,2.99,2.01,164.76,146.0,-47.59
4,366,366 Vincentina (A893 FG),14:48,15:12,15:37,00:49,22:58:23,"-07 19'09""",12.8,2.98,1.97,175.26,136.6,-56.52
5,69,69 Hesperia (A861 HC),13:39,15:13,16:48,03:08,22:59:20,"-01 58'30""",11.7,3.23,2.22,175.20,133.7,-53.09
6,545,545 Messalina (A904 TC),13:39,15:29,17:18,03:39,23:14:40,"+00 08'02""",12.5,2.77,1.76,175.00,129.3,-54.08
7,51,51 Nemausa (A858 BA),14:00,15:34,17:09,03:08,23:20:38,"-01 59'00""",10.9,2.52,1.52,176.76,129.2,-56.66
8,800,800 Kressmannia (A915 FG),13:47,15:40,17:32,03:45,23:25:45,"+00 35'42""",12.6,1.81,0.807,173.88,126.8,-55.39
9,709,709 Fringilla (A911 CC),13:03,15:48,18:33,05:30,23:34:02,"+11 42'38""",12.8,2.59,1.61,162.74,118.8,-46.85


여기부터 승민이가 ~~~

In [13]:
from pathlib import Path
from astroquery.mpc import MPC
from pprint import pprint
import re

for _, row  in df.iterrows():
    print("row['Full name'] :", row["Full name"])
    
    #str ="LOG_ADD(LOG_FLOAT, actuatorThrust, &actuatorThrust)"
    asteroid_names = re.findall('\(([^)]+)', row['Full name'])   #extracts string in bracket()
    print (asteroid_names)
    
    eph = MPC.get_ephemeris(asteroid_names[0],  step='1h', number=30 * 24)
    print(f"Proper motion: {eph['Proper motion'].quantity.to('arcmin/h').max():.03f}")
    eph = MPC.get_ephemeris(asteroid_names[0], location='P64',
                        ra_format={'sep': ':', 'unit': 'hourangle', 'precision': 1}, 
                        dec_format={'sep': ':', 'precision': 0},
                        step='1h')
    print("type(eph): ", type(eph))
    #print("eph: ", eph) 
    df_eph = eph.to_pandas() 
    print("df_eph :", df_eph)
    break

row['Full name'] : 393 Lampetia (A894 VC)
['A894 VC']
Proper motion: 1.104 arcmin / h
type(eph):  <class 'astropy.table.table.Table'>
df_eph :                   Date          RA       Dec  Delta      r  Elongation  Phase  \
0  2023-09-12 11:00:00  19:02:04.5   0:17:22  1.206  1.874       115.4   29.0   
1  2023-09-12 12:00:00  19:02:07.1   0:16:58  1.206  1.874       115.4   29.0   
2  2023-09-12 13:00:00  19:02:09.8   0:16:33  1.207  1.874       115.4   29.0   
3  2023-09-12 14:00:00  19:02:12.4   0:16:08  1.207  1.874       115.3   29.0   
4  2023-09-12 15:00:00  19:02:15.1   0:15:43  1.207  1.874       115.3   29.1   
5  2023-09-12 16:00:00  19:02:17.8   0:15:19  1.208  1.874       115.3   29.1   
6  2023-09-12 17:00:00  19:02:20.5   0:14:54  1.208  1.874       115.3   29.1   
7  2023-09-12 18:00:00  19:02:23.3   0:14:29  1.208  1.874       115.2   29.1   
8  2023-09-12 19:00:00  19:02:26.1   0:14:05  1.209  1.874       115.2   29.1   
9  2023-09-12 20:00:00  19:02:28.9   0:13:40  1

In [ ]:
# from astropy.time import Time
# import astropy.units as u
# from astroquery.mpc import MPC

# eph = MPC.get_ephemeris(OBJID, location='P64', 
#                         start=Time(hdul[0].header["DATE-OBS"]).strftime('%Y-%m-%d %H:%M:%S'),
#                         number=1,
#                         )
# print("type(eph) :", type(eph))
# print("eph :", eph)

# import pandas as pd
# df_eph = eph.to_pandas()
# df_eph

In [14]:
eph = MPC.get_ephemeris('A894 VC', start='1996-03-01', step='1h', number=30 * 24)
print(eph['Proper motion'].quantity.to('deg/h').max())
eph = MPC.get_ephemeris('A894 VC', location='P64',
                        ra_format={'sep': ':', 'unit': 'hourangle', 'precision': 1}, 
                        dec_format={'sep': ':', 'precision': 0},
                        step='1h')
print(eph)    

0.023877777777777776 deg / h
          Date              RA       Dec    ... Uncertainty 3sig Unc. P.A.
                        hourangle    deg    ...      arcsec         deg   
----------------------- ---------- -------- ... ---------------- ---------
2023-09-12 11:00:00.000 19:02:04.5  0:17:22 ...                0      74.5
2023-09-12 12:00:00.000 19:02:07.1  0:16:58 ...                0      74.5
2023-09-12 13:00:00.000 19:02:09.8  0:16:33 ...                0      74.5
2023-09-12 14:00:00.000 19:02:12.4  0:16:08 ...                0      74.5
2023-09-12 15:00:00.000 19:02:15.1  0:15:43 ...                0      74.6
2023-09-12 16:00:00.000 19:02:17.8  0:15:19 ...                0      74.6
2023-09-12 17:00:00.000 19:02:20.5  0:14:54 ...                0      74.6
2023-09-12 18:00:00.000 19:02:23.3  0:14:29 ...                0      74.5
2023-09-12 19:00:00.000 19:02:26.1  0:14:05 ...                0      74.6
2023-09-12 20:00:00.000 19:02:28.9  0:13:40 ...                0      7